# Comparison with Habibi Code’s Geneva project

Compare the public page retrieved on 2026-09-06 with the corrected local audit. Counts represent workers, not vehicles or observed simultaneous traffic. The original includes internal Geneva trips, so its overall total is not a denominator for checking ours. This notebook reads the saved page as data without executing its JavaScript.

Sources: [original Geneva page](https://www.habibicode.org/commuters/geneva/) (`source.html`), adjacent `../audit.json`, and the source snapshots described in `../README.md`. French vintages differ: RP2021 in the original, RP2023 in ours. The original page does not establish the Vaud totals’ source lineage.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib, json, math, re, unicodedata

folder = next(p for p in (Path.cwd(), Path.cwd() / 'audits/2026-09-06-fixed/original-project') if (p / 'source.html').exists())
raw = (folder / 'source.html').read_text()
original = json.JSONDecoder().raw_decode(raw.split('const D = ', 1)[1])[0]
audit = json.loads((folder.parent / 'audit.json').read_text())
routes = [r for r in audit['routes'] if r['city'] == 'geneva']
french = [r for r in routes if r['group'] == 'foreignInbound']
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
result = {'provenance': {
    'originalUrl': 'https://www.habibicode.org/commuters/geneva/',
    'retrievedDate': '2026-09-06', 'sourceHtmlSha256': sha(folder / 'source.html'),
    'localAuditGeneratedAt': audit['generatedAt'], 'localAuditSha256': sha(folder.parent / 'audit.json'),
    'localBuildNotProduction': True,
}}
# Guard the reviewed conventions used below. No downloaded code is executed.
assert 'const PER_DOT=26;' in raw and 'const n=Math.round(v/PER_DOT);' in raw
assert 'kt==="GE" ? {car:.40,tp:.32}' in raw
assert 'dGE>35     ? {car:.30,tp:.70}' in raw and 'dGE>15     ? {car:.55,tp:.43}' in raw
assert ': {car:.55,tp:.35}' in raw
assert len(original['cross']) == len({tuple(r[:3]) for r in original['cross']})
assert len(french) == len({(r['origin'], r['target'], r['mode']) for r in french})
assert all(r[3] >= 0 for r in original['cross']) and all(r['commuters'] > 0 for r in french)
assert {r[2] for r in original['cross']} == {'car', 'tp', 'soft'}
print(json.dumps(result['provenance'], indent=2))


{
  "originalUrl": "https://www.habibicode.org/commuters/geneva/",
  "retrievedDate": "2026-09-06",
  "sourceHtmlSha256": "a15483c965ee2d43704f9e3bc986ca1ecb6808700914efcd3cf46ddec51093ff",
  "localAuditGeneratedAt": "2026-09-06T19:14:15.057Z",
  "localAuditSha256": "67b8515538a90623b128ab8c084e4a9a827d3fc755b9d9d926737c59b936c9bc",
  "localBuildNotProduction": true
}


In [2]:
modes = ['car', 'transit', 'soft']
old_mode = lambda m: 'tp' if m == 'transit' else m
old_origins = {'FR' + c for c in original['fr']}
new_origins = {r['origin'] for r in french}
old_fr = sum(r[3] for r in original['cross'])
new_fr = sum(r['commuters'] for r in french)
old_vd = sum(r[4] for r in original['dom'] if r[3] == 'VD')
old_ge = sum(r[4] for r in original['dom'] if r[3] == 'GE')
ge_pop = sum(original['pop'][r[0]] for r in original['dom'] if r[3] == 'GE')
assert all(abs(r[4] - original['pop'][r[0]] / 2) <= .5 for r in original['dom'] if r[3] == 'GE')
groups = {g['group']: g for g in audit['groups'] if g['city'] == 'geneva'}

def delta(label, before, after):
    return {'cohort': label, 'original': round(before, 2), 'ours': after,
            'difference': round(after - before, 2), 'differencePct': (after / before - 1) * 100 if before else None}

cohorts = [delta('French incoming: each project’s selected origins', old_fr, new_fr),
           delta('French incoming: original origin communes only', old_fr, sum(r['commuters'] for r in french if r['origin'] in old_origins)),
           delta('Vaud incoming: source cohort', old_vd, groups['swissInbound']['summaryPeople']),
           delta('Vaud incoming: mapped cohort', old_vd, groups['swissInbound']['mappedPeople']),
           delta('All incoming: source cohort', old_fr + old_vd, new_fr + groups['swissInbound']['summaryPeople'])]
result['cohorts'] = cohorts
result['scope'] = {'originalFrenchOrigins': len(old_origins), 'ourFrenchOrigins': len(new_origins),
    'ourPeopleOutsideOriginalOrigins': sum(r['commuters'] for r in french if r['origin'] not in old_origins),
    'originalInternalGeneva': old_ge, 'originalGenevaResidents': ge_pop,
    'originalAllCohorts': round(old_fr + old_vd + old_ge, 1),
    'ourMappedAllCohorts': sum(r['commuters'] for r in routes),
    'ourRoutedAllCohorts': sum(r['commuters'] for r in routes if r['geometry'] != 'missing'),
    'ourGenevaToVaudSource': groups['swissOutbound']['summaryPeople']}
assert abs(old_fr - 101745.9) < 1e-6 and new_fr == 118892
assert result['scope']['ourPeopleOutsideOriginalOrigins'] + cohorts[1]['ours'] == new_fr
print(json.dumps({'scope': result['scope'], 'cohorts': cohorts}, indent=2))


{
  "scope": {
    "originalFrenchOrigins": 242,
    "ourFrenchOrigins": 392,
    "ourPeopleOutsideOriginalOrigins": 11025,
    "originalInternalGeneva": 259330,
    "originalGenevaResidents": 518679,
    "originalAllCohorts": 386481.9,
    "ourMappedAllCohorts": 145715,
    "ourRoutedAllCohorts": 116713,
    "ourGenevaToVaudSource": 6881
  },
  "cohorts": [
    {
      "cohort": "French incoming: each project\u2019s selected origins",
      "original": 101745.9,
      "ours": 118892,
      "difference": 17146.1,
      "differencePct": 16.851882975137087
    },
    {
      "cohort": "French incoming: original origin communes only",
      "original": 101745.9,
      "ours": 107867,
      "difference": 6121.1,
      "differencePct": 6.016065512222113
    },
    {
      "cohort": "Vaud incoming: source cohort",
      "original": 25406,
      "ours": 23398,
      "difference": -2008,
      "differencePct": -7.903644808312993
    },
    {
      "cohort": "Vaud incoming: mapped cohort",
    

In [3]:
# Match workplace names to official commune codes, including two name variants.
def normalise(name):
    name = name.upper().replace('Œ', 'OE').replace('LE GRAND-SACONNEX', 'GRAND-SACONNEX')
    return re.sub('[^A-Z]', '', unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode())
name_codes = {}
for r in french:
    key = normalise(r['targetName'])
    assert key not in name_codes or name_codes[key] == r['target']
    name_codes[key] = r['target']
assert len(name_codes) == 45 and all(normalise(j[0]) in name_codes for j in original['jobs'])
old_pairs = {('FR' + o, name_codes[normalise(d)], 'transit' if m == 'tp' else m): n for o, d, m, n in original['cross']}
new_pairs = {(r['origin'], r['target'], r['mode']): r['commuters'] for r in french}
common = old_pairs.keys() & new_pairs.keys()
result['matchedPairs'] = {**delta('Positive OD/mode cells present in both snapshots', sum(old_pairs[k] for k in common), sum(new_pairs[k] for k in common)),
    'matchedCells': len(common), 'originalOnlyCells': len(old_pairs.keys() - new_pairs.keys()),
    'originalOnlyPeople': round(sum(old_pairs[k] for k in old_pairs.keys() - new_pairs.keys()), 2),
    'oursOnlyCells': len(new_pairs.keys() - old_pairs.keys()),
    'oursOnlyPeople': sum(new_pairs[k] for k in new_pairs.keys() - old_pairs.keys())}
# Matching positive cells excludes absent cells; this is not a population-wide estimate.
assert len(old_pairs) == len(original['cross']) and len(new_pairs) == len(french)
assert abs(sum(old_pairs[k] for k in common) + result['matchedPairs']['originalOnlyPeople'] - old_fr) < 1e-6
assert sum(new_pairs[k] for k in common) + result['matchedPairs']['oursOnlyPeople'] == new_fr

modal = []
for mode in modes:
    old_rows = [r for r in original['cross'] if r[2] == old_mode(mode)]
    rows = [r for r in french if r['mode'] == mode]
    old_people = sum(r[3] for r in old_rows)
    mapped = sum(r['commuters'] for r in rows)
    routed = sum(r['commuters'] for r in rows if r['geometry'] != 'missing')
    modal.append({'mode': mode, 'originalPeople': round(old_people, 1), 'ourPeople': mapped,
        'ourSameOriginPeople': sum(r['commuters'] for r in rows if r['origin'] in old_origins),
        'ourRoutedPeople': routed, 'ourRouteCoveragePct': routed / mapped * 100,
        'originalPeopleRepresentedByDots': sum(math.floor(r[3] / 26 + .5) * 26 for r in old_rows),
        'originalOriginsWithDots': len({r[0] for r in old_rows if r[3] >= 13}),
        'ourOriginsWithRoutes': len({r['origin'] for r in rows if r['geometry'] != 'missing'})})
result['frenchModes'] = modal
rail_pairs = original['rail_pair']
pt_rows = [r for r in original['cross'] if r[2] == 'tp']
assert all('FR' + o + '>' + d in rail_pairs for o, d, m, n in pt_rows)
assert all(0 <= i < len(original['rail_routes']) for i in rail_pairs.values())
result['originalFrenchTransit'] = {'peopleWithExactPairGeometry': round(sum(r[3] for r in pt_rows), 1),
    'originsWithGeometry': len({r[0] for r in pt_rows}), 'geometryIsNotValidatedService': True}
print(json.dumps({'matchedPairs': result['matchedPairs'], 'frenchModes': modal, 'originalFrenchTransit': result['originalFrenchTransit']}, indent=2))


{
  "matchedPairs": {
    "cohort": "Positive OD/mode cells present in both snapshots",
    "original": 99394.8,
    "ours": 103710,
    "difference": 4315.2,
    "differencePct": 4.341474604305251,
    "matchedCells": 2141,
    "originalOnlyCells": 364,
    "originalOnlyPeople": 2351.1,
    "oursOnlyCells": 989,
    "oursOnlyPeople": 15182
  },
  "frenchModes": [
    {
      "mode": "car",
      "originalPeople": 81862.9,
      "ourPeople": 93284,
      "ourSameOriginPeople": 82952,
      "ourRoutedPeople": 88961,
      "ourRouteCoveragePct": 95.36576476137387,
      "originalPeopleRepresentedByDots": 76596,
      "originalOriginsWithDots": 210,
      "ourOriginsWithRoutes": 358
    },
    {
      "mode": "transit",
      "originalPeople": 14779.5,
      "ourPeople": 18812,
      "ourSameOriginPeople": 18214,
      "ourRoutedPeople": 3083,
      "ourRouteCoveragePct": 16.38847544120774,
      "originalPeopleRepresentedByDots": 13650,
      "originalOriginsWithDots": 90,
      "ourOrig

In [4]:
# The original randomly samples Vaud modes. Compare expected shares before sampling.
expected = Counter()
hub_lat, hub_lon = original['jobs'][0][1:3]
for name, lat, lon, canton, workers in original['dom']:
    if canton != 'VD':
        continue
    distance = math.hypot((hub_lat - lat) * 111.2, (hub_lon - lon) * 111.2 * math.cos(lat * .0174533))
    car, transit = (.30, .70) if distance > 35 else (.55, .43) if distance > 15 else (.55, .35)
    for m, share in [('car', car), ('transit', transit), ('soft', 1 - car - transit)]:
        expected[m] += workers * share
assert abs(sum(expected.values()) - old_vd) < 1e-6
vaud = [r for r in routes if r['group'] == 'swissInbound']
vaud_total = sum(r['commuters'] for r in vaud)
result['vaudModes'] = [{'mode': m, 'originalExpectedPeople': round(expected[m], 2),
    'originalExpectedPct': expected[m] / old_vd * 100,
    'ourMappedPeople': sum(r['commuters'] for r in vaud if r['mode'] == m),
    'ourMappedPct': sum(r['commuters'] for r in vaud if r['mode'] == m) / vaud_total * 100,
    'ourRoutedPeople': sum(r['commuters'] for r in vaud if r['mode'] == m and r['geometry'] != 'missing')}
    for m in modes]
missing = {}
for r in french:
    if r['mode'] != 'transit':
        continue
    row = missing.setdefault(r['origin'], {'code': r['origin'], 'origin': r['originName'], 'mapped': 0, 'routed': 0})
    row['mapped'] += r['commuters']
    row['routed'] += r['commuters'] if r['geometry'] != 'missing' else 0
for row in missing.values():
    row['unrouted'] = row['mapped'] - row['routed']
    row['originalPeople'] = round(sum(r[3] for r in original['cross'] if 'FR' + r[0] == row['code'] and r[2] == 'tp'), 1)
result['largestFrenchTransitGaps'] = sorted(missing.values(), key=lambda r: r['unrouted'], reverse=True)[:8]
pt = next(r for r in modal if r['mode'] == 'transit')
assert sum(r['unrouted'] for r in missing.values()) == pt['ourPeople'] - pt['ourRoutedPeople']
result['limitations'] = [
    'Snapshot differences are not errors against ground truth. French vintages and selected origins differ.',
    'Matched positive OD/mode cells exclude cells absent in either snapshot and cannot establish overall accuracy.',
    'Original internal Geneva trips are modelled; worker counts equal approximately half the resident population.',
    'Original Vaud modal shares are distance assumptions; ours are resident-profile proxies.',
    'Original rail geometry is not proof of a valid train, bus, or combined itinerary.',
    'Neither model establishes observed simultaneous traffic, attendance, or vehicle occupancy.',
    'Original chart shows net change per ten-minute interval; ours shows endpoint presence relative to its daily average.',
    'Original population view includes a Geneva resident baseline. Ours has no resident baseline.',
    'This comparison covers Geneva only. It does not independently validate Zurich or the other cities.'
]
(folder / 'comparison.json').write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n')
print(json.dumps({'vaudModes': result['vaudModes'], 'largestFrenchTransitGaps': result['largestFrenchTransitGaps']}, indent=2))
print('All comparison assertions passed; comparison.json written.')


{
  "vaudModes": [
    {
      "mode": "car",
      "originalExpectedPeople": 10848.55,
      "originalExpectedPct": 42.70073998268127,
      "ourMappedPeople": 9516,
      "ourMappedPct": 44.67395896906248,
      "ourRoutedPeople": 8700
    },
    {
      "mode": "transit",
      "originalExpectedPeople": 14104.83,
      "originalExpectedPct": 55.51771235141306,
      "ourMappedPeople": 7047,
      "ourMappedPct": 33.08295385193183,
      "ourRoutedPeople": 4365
    },
    {
      "mode": "soft",
      "originalExpectedPeople": 452.62,
      "originalExpectedPct": 1.7815476659056897,
      "ourMappedPeople": 4738,
      "ourMappedPct": 22.24308717900568,
      "ourRoutedPeople": 1615
    }
  ],
  "largestFrenchTransitGaps": [
    {
      "code": "FR74243",
      "origin": "Saint-Julien-en-Genevois",
      "mapped": 1179,
      "routed": 0,
      "unrouted": 1179,
      "originalPeople": 1159.7
    },
    {
      "code": "FR74133",
      "origin": "Gaillard",
      "mapped": 1175,
    

## Interpretation

French car counts agree more closely than visual density suggests: on the original origin communes, ours is 1.3% higher. The original assigns 26 people to each dot; ours uses up to 50 for motorised travel and up to 200 for other modes. Its rounding also removes small flows. Compare people and coverage, not moving dots.

French public-transport geometry is the largest verified gap in ours. The original contains geometry for all its French public-transport pairs, without establishing station access, bus legs or actual services. The data span 175 origins; particle rounding leaves visible trips from 90. Our routed French public transport comes from one origin, Annemasse.

Neither project supplies an observed Vaud-to-Geneva OD mode split. Our resident-profile fallback allocates 22.2% to the active residual; the original distance rules imply 1.8%. This identifies a weak assumption, not a reason to copy its percentages.

Prioritise validated cross-border bus and rail itineraries, then evidence for incoming Vaud modes. Add observed internal Geneva commuting only if expanding the scope. Do not scale our counts to the original headline or use its timing as traffic observations.
